# Q-Learning  
### *Model-free RL, off-policy method*

#### Load the Tic-Tac-Toe environment.

In [1]:
from tic_tac_toe_env import TicTacToe
import random

#### Load opponent policies.

In [2]:
def random_move(game: TicTacToe, letter: str):
    board = game.board
    player = letter
    opponent = 'O' if player == 'X' else 'X'

    move = random.choice(game.available_moves())
    return move

In [5]:
def choose_move(game: TicTacToe, letter: str):
    # board = list(game.get_flat_state())
    board = game.board
    n = game.n
    player = letter
    opponent = 'O' if player == 'X' else 'X'

    def empty():
        return game.available_moves()

    def lines():
        all_lines = []
        for i in range(n):
            all_lines.append([i*n+j for j in range(n)])
        for j in range(n):
            all_lines.append([i*n+j for i in range(n)])
        # diag
        all_lines.append([i*n+i for i in range(n)])
        # off-diag
        all_lines.append([i*n+(n-1-i) for i in range(n)])
        return all_lines

    def can_win(marker):
        winning_moves = []
        for line in lines():
            vals = [board[i] for i in line]
            if vals.count(marker) == n-1 and vals.count('_') == 1:
                winning_moves.append(line[vals.index('_')])
        return winning_moves # return the list of indicies for winning moves

    # if our wins is not empty then play any of those moves to win (here just pick the first)
    our_wins = can_win(player)
    if our_wins:
        return our_wins[0]

    # if the opponent can win then we just block the move. From the lecture we should just pick it randomly
    opp_wins = can_win(opponent)
    if opp_wins:
        return np.random.choice(opp_wins).item()
    
    empty_cells = empty()

    first_empty = empty_cells[0]
    
    # play sequentially in the row first
    current_row = first_empty//n # recall that these are flattened indices
    next_in_row = first_empty + 1 # next cell in the same row
    
    # if its truly on the same row and empty then play it, otherwise it might wrap around and not make snese
    if next_in_row < (current_row + 1) * n and next_in_row in empty_cells:
        return next_in_row
    
    # now, if thats the case, then try the cell below
    cell_below = first_empty + n
    if cell_below < n*n and cell_below in empty_cells: # so if its actually valid (which it should be) and its empty then play it
        return cell_below
    
    # else play randomly
    return np.random.choice(empty_cells).item()

#### Setup the Q-Learning Agent.

In [ ]:
import random
import copy
import pickle

class QLearningAgent:
    def __init__(self, alpha=0.5, gamma=1.0, epsilon=0.1):
        self.q_table = {}  # (state_tuple, action) -> Q-value
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon

    def get_q(self, state, action):
        return self.q_table.get((state, action), 0.0)

    def choose_action(self, game, letter):
        state = tuple(game.board)
        available = game.available_moves()
        if random.random() < self.epsilon:
            return random.choice(available)
        # Pick action with max Q-value
        q_values = [self.get_q(state, a) for a in available]
        max_q = max(q_values)
        max_actions = [a for a, q in zip(available, q_values) if q == max_q]
        return random.choice(max_actions)

    def learn(self, state, action, reward, next_state, done, game, nextActions = None):
        old_q = self.get_q(state, action)
        # future_q = 0 if done else max([self.get_q(next_state, a) for a in range(9)])
        # available = game.available_moves()
        if done or not nextActions:
            target = reward
        else:
             # future_q = 0 if done else max(self.get_q(next_state, a) for a in available)
            future_q = max(self.get_q(next_state, a) for a in nextActions)
            target = reward + self.gamma * future_q

        self.q_table[(state, action)] = old_q + self.alpha * (target - old_q)

def train(agent, episodes=50000, boardsize=3):
    for _ in range(episodes):
        game = TicTacToe(n=boardsize)
        state = tuple(game.board)
        letter = 'X'

        
        while game.empty_squares():
            action = agent.choose_action(game, letter)
            game.make_move(action, letter)
            next_state = tuple(game.board)
            done = game.current_winner is not None or not game.empty_squares()
            next_actions = game.available_moves() if not done else []

            if done:
                # Winner gets +1, loser -1
                if game.current_winner:
                    reward = 1 if letter == game.current_winner else -1
                else:
                    reward = 0
            else:
                reward = 0

            agent.learn(state, action, reward, next_state, done, game, next_actions)
            state = next_state
            letter = 'O' if letter == 'X' else 'X'


In [4]:
def play_human(agent, board_size, human_symbol='X'):
    game = TicTacToe(n=board_size)
    human_turn = human_symbol == 'X'

    while game.empty_squares():
        game.print_board()
        if human_turn:
            move = int(input("Enter your move (0-8): "))
            if move not in game.available_moves():
                print("Invalid move, try again.")
                continue
            game.make_move(move, human_symbol)
        else:
            move = agent.choose_action(game, 'O')
            game.make_move(move, 'O')
            print(f"AI plays: {move}")

        if game.current_winner:
            game.print_board()
            winner = "Human" if human_turn else "AI"
            print(f"{winner} wins!")
            return
        human_turn = not human_turn

    game.print_board()
    print("It's a tie!")

#### Train the Q-Learning Agent

In [50]:
# Usage:
#5000000
agent = QLearningAgent()
train(agent, episodes=10000000, boardsize=4)

In [47]:
import pickle
with open("DEC6_q_learning_4x4_eps=8000000", "wb") as f:
  pickle.dump(agent.q_table, f)

In [48]:
with open("DEC6_q_learning_4x4_eps=8000000", "rb") as f:
  agent.q_table = pickle.load(f)
  print(f"Q-values loaded")

Q-values loaded


#### Enable self play against human.

In [49]:
play_human(agent,4)

|   |   |   |   |
|   |   |   |   |
|   |   |   |   |
|   |   |   |   |
| X |   |   |   |
|   |   |   |   |
|   |   |   |   |
|   |   |   |   |
AI plays: 7
| X |   |   |   |
|   |   |   | O |
|   |   |   |   |
|   |   |   |   |
| X | X |   |   |
|   |   |   | O |
|   |   |   |   |
|   |   |   |   |
AI plays: 5
| X | X |   |   |
|   | O |   | O |
|   |   |   |   |
|   |   |   |   |
| X | X | X |   |
|   | O |   | O |
|   |   |   |   |
|   |   |   |   |
AI plays: 3
| X | X | X | O |
|   | O |   | O |
|   |   |   |   |
|   |   |   |   |
| X | X | X | O |
| X | O |   | O |
|   |   |   |   |
|   |   |   |   |
AI plays: 12
| X | X | X | O |
| X | O |   | O |
|   |   |   |   |
| O |   |   |   |
| X | X | X | O |
| X | O | X | O |
|   |   |   |   |
| O |   |   |   |
AI plays: 11
| X | X | X | O |
| X | O | X | O |
|   |   |   | O |
| O |   |   |   |
Invalid move, try again.
| X | X | X | O |
| X | O | X | O |
|   |   |   | O |
| O |   |   |   |
| X | X | X | O |
| X | O | X | O |
|   |   |   |

ValueError: invalid literal for int() with base 10: ''